## MaMa course practical


## Introduction

The goal of the project is to explores how sediment temperature varies across space and time in the intertidal zone of the Dutch Wadden Sea. In this document, you will be tasked with analyzing high frequency datasets you derived from your field work and develop a conceptual and simple model of the sediment heat budget over tidal cycle, sediment type(s) and/or inundation gradient.

## TempSED - Sediment temperature model

This document contains snippet of code to get you started in modeling sediment temperature using the TempSED. Basically, the model aim to simulate the temperature of the bulk sediment as a function of depth and time.to simulate the temperature of the bulk sediment as a function of depth and time. For the mathematically incline of you, the model essentially, we are solving the following partial differential equation. Don't worry, you will not solve the equation analytically. We will let computer do the work for us.

$$\frac{\partial T_{bs}}{\partial t} = \frac{1}{c_{bs} \cdot \rho_{bs}} \cdot \frac{\partial }{\partial x}(K_{bs} \cdot \frac{\partial T_{bs}}{\partial x}) - \frac{1}{c_{bs} \cdot \rho_{bs}} \cdot \frac{\partial I}{\partial x}$$


where $x$ and $t$ denote depth in the sediment and time, respectively. In this equation, the first term on the right-hand side represents the heat flow due to temperature gradients, while the second term represents the heat source due to attenuation of the incident short-wave solar radiation (I , in W m−2) in the sediment. Furthermore, the parameters $k_{bs}$  $c_{bs}$ and $\rho_{bs}$ denote the thermal conductivity (W m−1 K−1), specific heat capacity (J kg−1 K−1) and density (kg m−3) of the bulk sediment, respectively. Because bulk sediment is a mixture of different types of solids and water, these parameters depend on the sediment porosity and thus, in general, vary with depth in the sediment.

with boundary conditions

$$\frac{\partial T_{bs}}{\partial x}|_{x=\infty} = 0,$$ at the lower boundary, and $$K _{bs} \cdot \frac{\partial T_{bs}}{\partial x}|_{x=0} =  H_0,$$ at the upper boundary, when exposed to the air, or $$T_{bs}|_{x=0}=T_{ow},$$ When exposed to the water.

Suffice to say, we will use a computer program developed at NIOZ to simulate how sediment temperature will response to varying atmospheric and water condition. To do that, first you need to install R and some external packages (In the lab, I can help you if you run into installation issues). Ideally, if would install an integrated development environment (IDE) called R-studio but you can do without it. Next, run the following code below to install `TempSED`:\



In [ ]:
req_dep_pkgs <- c("deSolve", "rootSolve", "ReacTran", "devtools", "remotes")

if(!all(req_pkgs %in% rownames(installed.packages()))){
  needed_pkgs <- req_pkgs[!req_pkgs %in% rownames(installed.packages())]
  install.packages(needed_pkgs)
} else {
  invisible(lapply(req_pkgs, require, character.only = TRUE))
}


devtools::install_github("TempSED/TempSED", depend=TRUE)



With the package installed, Function *TempSED_run1D* implements the one-dimensional mechanistic model for sediment temperature in the vertical dimension. It is a numerical model, where the sediment is subdivided in 100 vertical boxes. The model units are $m$, $s$, $^oC$. See appendix for the model equations.

The function is defined as:


In [ ]:


```{r}
#| echo: false
#| comment: NA

Def_fun(TempSED_run1D)
```



where

- *z_max*, *dz\_.\_1* are the total sediment depth and the size of the first box, in $m$,
- *porosity* is the sediment porosity,
- *T_ini* is the initial condition; times represents the output times for the model,
- *f_Waterheight*, *f_Watertemperature*, the overlying water conditions
- *f_Airtemperature*, *f_Qrel*, *f_Pressure*, *f_Solarradiation*, *f_Windspeed*, *f_Cloudiness* are the atmospheric conditions,
- *irrigation* is the sediment irrigation rate,
- *sedpos*, if not NULL, is a vector with sediment depths whose temperature needs to be outputted.

The forcings are either a time series, a function that takes an argument time, or one value.

the following parameters can be defined in *parms*:

- *cp_water*, *cp_solid* in $J/kg/K$, *tc_water*, *tc_solid* in $W/m/K$ and *density_water*, *density_solid* in $kg/m3$, is the specific heat capacity (cp..), the thermal conductivity (tc..), and the density (dens..) for water and the solid fraction respectively.
- *albedo_water*, *albedo_sediment*, *kd_water*, *kd_sediment* are the albedo (reflectance, $-$) and light extinction coefficient ($/m$) for water and sediment respectively,
- *em_air*, *em_sediment*, are the emissivity of air and bulk sediment respectively,

Arguments *porosity*, *irrigation* and *T_ini* can either be one value or a function that takes as an argument the sediment depth. *T_ini* can also be the output of a previous run, if the model needs to be restarted.

### Your first simple model

For this simple model, we will simulate the sediment temperature up to maximum depth of 10m, with thickness of 0.04 cm (i.e 0.0004 m). For simplicity, we will imposed a constant porosity of 0.9 but this can be changed to described how porosity will change with depth. We will assumed the air temperature is 15 degree and a solar radiation at high afternoon of 270 W/m2. We will run for the period of MaMa course (2 weeks).


In [ ]:

z_max    <- 10
dz_1     <- 1e-4
length.z <- 10
porosity  <- 0.9
fAirTempYr <- 15 
fSolarRadYr <- 270
days_to_run <- 14 

times <- seq(from = 0, to = days_to_run*86400, by = 86400)

simple_model <-TempSED_run1D(
  z_max =z_max,
  dz_1 =dz_1,
  porosity =porosity,
  times =times,
  T_ini =10,
  f_Airtemperature=fAirTempYr,
  f_Solarradiation=fSolarRadYr)



Now we can visualize our result:

In [ ]:
T_air<-subset(simple_model,which="Airtemperature")
image2D(simple_model, main = "Temperature", xlab = "days",
        clab = "degC", ylab = "m", time_unit = "day",
        contour = TRUE, mfrow = NULL)


### Constructing a realistic sediment temperature model

Now, that is simple enough, and hopefully give you an impression on how the model works. Now, you will build upon this to construct a realistic model driven by the data you obtained from the field.

For that, we need data to `Force` the model, `boundary conditions` to enforce exchange on energy and heat between the sediment and its overlying media and `observed data` to calibrate/validate the model.

Speaking of observed data, we can first take a look at the task at hand. What kind of profile are we to expect here. With one of your field data, we can read it and visualize the sediment temperature.


In [ ]:
data_22026091 <- read.csv("~/../../OneDrive - NIOZ/Desktop/Digital_twin/Student-Thesis/MaMa course/MaMa_course_demo_data/2024_LoggerOutput/2024_LoggerOutput/22026091.csv", skip = 1)

data_22026091 <- data_22026091[, 2:3]
names(data_22026091) <- c("date", "temperature")
data_22026091$date <- as.POSIXct(data_22026091$date, format = "%m/%d/%Y %I:%M:%S %p", tz = "GMT")
head(data_22026091)

with(data_22026091, plot(date, temperature, type = "l", ylab = "Temperature (dgC)", xlab = "Date"))


### Atmospheric forcing

Beyond the observed data, our model need some atmospheric data that drive changes we observed in the sediment. Important meteorological data are crucial to properly simulate how the sediment will response at different timescale. For example, on a warm day, we expect higher air temperature and high solar radiation which *can* propagate heat to the sediment, however, things are not so simple as you would as other factors like windspeed, atmospheric pressure, sky albedo (cloudiness) can affect how much heat is exchanged on the mudflat. So, we need to take all this into account.

For that, you will need to get your hands on some meteorological data from KNMI. Head over to this website (KNMI) \[<https://daggegevens.knmi.nl/klimatologie/uurgegevens>\] and grab some data in one of the several stations closest to your sampling location.

In [ ]:
### Load your data here 

devtools::load_all("~/../../home/lter_life_dev/git_repos/dtR/dtRtools/")

atmospheric_forcing <- read_KNMI(file = "KMNI_Wad_2017.txt", dir = "~/../../home/lter_life_dev/KMNI/")
atmospheric_forcing$time_cest <- as.POSIXct(format(atmospheric_forcing$datetime, tz = "CET", usetz = TRUE))

start_date <- sort(atmospheric_forcing$time_cest)[1]
atmospheric_forcing$second <- as.numeric(difftime(atmospheric_forcing$time_cest, start_date, units = "secs"))

atmospheric_forcing$pressure <- atmospheric_forcing$pressure * 100 # from Hpa to pa
atmospheric_forcing$cloudcover <- atmospheric_forcing$cloudcover/8 # from 0-8 to 0-1 
## Select station closer to Denhelder
Dekooy <- subset(atmospheric_forcing, which = station == 235) # De Kooy
Dekooy <- Dekooy[order(Dekooy$datetime) & !is.na(Dekooy$datetime), ]

fWind_wad       <- Dekooy[, c("second", "windspeed")]
fRad_wad        <- Dekooy[, c("second", "radiation")]
fTair_wad       <- Dekooy[, c("second", "temperature")]
fPair_wad       <- Dekooy[, c("second", "pressure")] 
fhumidity_wad   <- Dekooy[, c("second", "humidity")]
fCloud_wad      <- Dekooy[, c("second", "cloudcover")]

In [ ]:
op <- par(mfrow = c(3, 2), mar = c(2, 5, 2, 2), las = 1, cex = 1.5)
plot(Dekooy$datetime, Dekooy$temperature, type = "l", ylab = "dgC", main = "Air temperature")
plot(Dekooy$datetime, Dekooy$windspeed, type = "l", ylab = "m/s", main = "Wind speed")
plot(Dekooy$datetime, Dekooy$cloudcover, type = "l", ylab = "-", main = "Cloud cover")
plot(Dekooy$datetime, Dekooy$radiation, type = "l", ylab = "W/m2", main = "Solar radition")
plot(Dekooy$datetime, Dekooy$pressure/1000, type = "l", ylab = "kPa", main = "Air pressure")
plot(Dekooy$datetime, Dekooy$humidity, type = "l", ylab = "-", main = "Air humidity")
par(op)


### Boundary and Initial conditions

In addition to the atmosphere, the sediment is mostly in contact with the water. However, this is phase dependent in the Wadden sea with strong tidal ebb and flow. Thus, the system (i.e sediment) is periodically exposed and submerged. Thus, properties of the overlying water changes with time (i.e water height, temperature and potentially salinity). This is crucially important as they modulate how much the sediment heats up.

To drive your model, you need an estimate of the water temperature and height at the site of collection. You can get some approximation of the water temperature and water height using measurement from Rijkwaterstraat sensors fixed at several locations in the Wadden Sea ()\[\].


In [ ]:
### Load your data 

RWS_parse <- read_RWS(dir = "~/../../home/lter_life_dev/RWS_Download/20240924_020/", 
                     file = "20240924_020.csv", 
                     format = "long")

plot(meta(RWS_parse)$stations$longitude, meta(RWS_parse)$stations$latitude, type = "p", col = 1, pch = 19)
text(meta(RWS_parse)$stations$longitude, meta(RWS_parse)$stations$latitude, label = meta(RWS_parse)$stations$station, cex = 0.5)

water_forcing <- subset(RWS_parse, subset = station == "OUDSD")

## For this dataset, we can subset for only year = 2017
water_forcing <- subset(water_forcing, subset = datetime >= as.POSIXct("2017-01-01") & datetime <= as.POSIXct("2017-12-31"))

water_forcing$second <- as.numeric(difftime(water_forcing$datetime, start_date, units = "secs"))
# water_forcing$second <- as.double(julian(water_forcing$datetime, origin = start_date) * 86400)

fHwater <- subset(water_forcing, subset = variable == "Height")[, c("second", "value")]

## For oudeschild, we do not have the corresponding water temperature
## So we use the data from nearby station 
nearby_water_forcing <- subset(RWS_parse, subset = station == "DOOVBWT")
nearby_water_forcing$second <- as.numeric(difftime(nearby_water_forcing$datetime, start_date, units = "secs"))
fTwater <- subset(nearby_water_forcing, subset = variable == "T")[, c("second", "value")]


find_time_comparison_function <- function(forcing_list) {
  # Initialize variables to store the latest start time
  # and earliest end time
  latest_start_time <- -Inf
  earliest_end_time <- Inf
  latest_start_function <- NULL
  earliest_end_function <- NULL
  # Loop over the list of functions and compare start and
  # end times
  for (function_name in names(forcing_list)) {
    df <- forcing_list[[function_name]]
    # Get the first and last value of the 'Second'
    # column
    first_second <- min(df$second, na.rm = TRUE)
    last_second <- max(df$second, na.rm = TRUE)
    # Check if this function starts later than the
    # current latest start time
    if (first_second > latest_start_time) {
      latest_start_time <- first_second
      latest_start_function <- function_name
    }
    # Check if this function ends earlier than the
    # current earliest end time
    if (last_second < earliest_end_time) {
      earliest_end_time <- last_second
      earliest_end_function <- function_name
    }
  }
  # Return the result as a named list
  list(LatestStartFunction = latest_start_function, FirstSecond = latest_start_time,
       EarliestEndFunction = earliest_end_function, LastSecond = earliest_end_time)
}


find_time_comparison_function(list(fwind = fWind_wad, frad = fRad_wad, ftair = fTair_wad, fpair = fPair_wad, 
                                   fhum = fhumidity_wad, 
                                   fcloud = fCloud_wad, fhwater = fHwater, ftwater = fTwater ))

### Model Runs

Now, you are ready to run the model using the data you have downloaded. Atmospheric forcing...check. Boundary conditions...check. What about parameters. In the sediment, there are lot of things going on. There are sediment being deposited and eroded periodically so the net effect is sediment is compacted, some worms and crabs are moving around and doing their wormy/crabby thing etc. All these effect need to **parameterized**. Depending on your background, we can go in-depth or simple. Here, we will start simple and depending on how good your model fit we can add some realism.

For a start, we can model sediment compaction using porosity. Porosity is a declining function from 0.9 near the SWI till 0.7 at depth.

In [ ]:
# function describing the variation of porosity (volume fraction of LIQUID) with depth
porosity_fun  <- function(x, por.SWI = 0.9, por.deep = 0.7, porcoef = 100){
  return( por.deep + (por.SWI-por.deep)*exp(-x*porcoef) )
}



Next, I provide a suite of parameters that you can affect an intertidal mudflat sediment heat dynamics over a day. Don't worry if there are a lot. Some are pretty insignificant and other are not. You will explore how sensitive they are in the last section.


In [ ]:
# parameters
parms <- list(em_air = 0.8, # [-] emissivity of air
              em_sediment = 0.95 * 1, # [-] emissivity of sediment
              stanton = 0.001 * 1, # [-] transfer coeff for sensible heat
              dalton = 0.0014 * 1, # [-] transfer coeff for latent heat
              density_water = 1024, # [kg/m3] seawater density
              density_solid = 2500, # [kg/m3] sediment dry density
              cp_water = 3994, # [J/kg/dgC] specific heat capacity water
              cp_solid = 700, # [J/kg/dgC] specific heat capacity solid (dry sediment)*
              tc_water = 0.6, # [W/m/dg] thermal conductivity of water
              tc_solid = 7, # [W/m/dg] thermal conductivity of solid* #1.68-19.21
              albedo_water = 0.05, # [-] part light reflected by water
              # albedo_sed = 0.1, # [-] part light reflected by sediment
              kd_water = 1, # [/m] light attenuation coefficient water (1-25/m in RT13)
              kd_sediment = 1000 # [/m] light attenuation coefficient (bulk) sediment
)

In [ ]:

times <- seq(from = 0, to = 30373380, by = 3600)

Tout_init <- TempSED_run1D(parms = parms, 
                      T_ini = 20, 
                      Grid = setup.grid.1D(N = 100, dx.1 = dz_1, L = length.z), 
                      times = times,
                      porosity = porosity_fun, 
                      dependency_on_T = FALSE,
                      f_Pressure = fPair_wad,
                      f_Airtemperature = fTair_wad, 
                      f_Waterheight = fHwater,
                      f_Watertemp = fTwater, 
                      f_Qrel  = fhumidity_wad,
                      f_Solarradiation = fRad_wad, 
                      f_Windspeed = fWind_wad,
                      f_Cloudiness = fCloud_wad, 
                      sedpos = c(0, 0.01, 0.03))

Tout <- TempSED_run1D(parms = parms, 
                      T_ini = Tout_init, 
                      Grid = setup.grid.1D(N = 100, dx.1 = dz_1, L = length.z), 
                      times = times,
                      porosity = porosity_fun, 
                      dependency_on_T = FALSE,
                      f_Pressure = fPair_wad,
                      f_Airtemperature = fTair_wad, 
                      f_Waterheight = fHwater,
                      f_Watertemp = fTwater, 
                      f_Qrel  = fhumidity_wad,
                      f_Solarradiation = fRad_wad, 
                      f_Windspeed = fWind_wad,
                      f_Cloudiness = fCloud_wad, 
                      sedpos = c(0, 0.01, 0.03))


### Model validation

Now we can validate how the model perform. We are going to plot the observed data from the field to the model and hopefully we can start to qweak things.

In [ ]:
model_time <- as.POSIXct(Tout[, "time"], origin = "2017-01-01 00:00:00") 
model_temp <- subset(Tout, which = "Tsed_mean") 
plot(model_time, model_temp, type = "l")

## replace with real year date
fake_date_time <- gsub("2024", "2017", data_22026091$date) |> as.POSIXct()
lines(fake_date_time, data_22026091$temperature, col = "red", lwd = 1)



Of course, this is not perfect. So we need to calibrate the model with physical/ecological basis in the next step.

### Sensitivity experiments
